# Entregável 10 — Validação Estatística Final do Dataset
 
**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Abril e Maio de 2026
 
---
 
## Objetivo
 
Este notebook realiza a **Validação Estatística Final do Dataset**, última etapa do pipeline de pré-processamento e preparação de dados antes da fase de Reconhecimento de Padrões (RP). O dataset de entrada é o subconjunto de features selecionadas no Entregável 9 (`features_selected.parquet`), que concentra os atributos com maior poder discriminativo e suporte estatístico formal.
 
O propósito desta etapa não é transformar os dados — eles já passaram por extração (E6), engenharia (E7) e seleção (E9). Vale notar que E8 (Redução deDimensionalidade) e E9 (Seleção de Atributos) são trilhas paralelas a partir do `features_engineered.parquet` do E7: o E8 produz o `features_pca.parquet`, uma representação comprimida destinada a classificadores que operam em espaço reduzido (kNN, SVM linear); o E9 opera sobre as features originais, preservando interpretabilidade clínica. O E10 usa a trilha do E9 por três razões objetivas: 

(i) componentes PCA são ortogonais por construção, tornandoa análise de VIF trivial e sem sentido; 

(ii) curvas de densidade e catálogode atributos exigem features com identidade de domínio (morfologia, HRV,wavelet etc.); 

(iii) o `features_pca.parquet` será consumido diretamente na fase de RP como entrada alternativa para classificadores específicos.

O objetivo aqui é **atestar formalmente** que o dataset chegou ao estado correto para ser entregue a qualquer classificador: sem redundância estrutural, com classes estatisticamente separáveis, balanceado ou com sua situação de desbalanceamento devidamente documentada, e com a distribuição de cada feature por classe rigorosamente mapeada.
 
Em termos práticos, este entregável responde a quatro perguntas antes de abrir o código de classificação:
 
1. **Multicolinearidade:** Existe colinearidade residual severa entre as features selecionadas? Features altamente colineares inflariam os coeficientes de modelos lineares e distorceriam a interpretação da importância de features em modelos baseados em árvore.
2. **Separabilidade estatística:** As classes (NORM, MI, STTC, CD, HYP) diferem significativamente em pelo menos um subconjunto das features? Isso valida que o pipeline gerou representações com real poder discriminativo.
3. **Balanceamento:** As classes estão representadas de forma suficientemente equilibrada no conjunto de treino? O desbalanceamento severo pode introduzir viés sistemático nos classificadores.
4. **Distribuição por classe:** Como se comporta a densidade de cada feature para cada classe? Curvas de densidade sobrepostas por classe revelam se a separação é clara, parcial ou praticamente inexistente — informação direta para guiar a escolha de classificadores e estratégias de regularização no RP.
 
O entregável está organizado em:
 
1. **Importações, Configurações e Dependências**
2. **Carregamento e Inspeção do Dataset Selecionado**
3. **Verificação de Multicolinearidade (VIF)**
   - 3.1 Cálculo do Variance Inflation Factor
   - 3.2 Análise do Heatmap de Correlação Residual
   - 3.3 Decisão e Remoção Iterativa de Features Colineares
4. **Separabilidade Estatística das Classes**
   - 4.1 Kruskal-Wallis por Feature
   - 4.2 Comparações Par-a-Par (Mann-Whitney U)
   - 4.3 Effect Size (Eta-quadrado e Cohen's d)
   - 4.4 Análise Discriminante Visual (Projeção LDA 2D)
5. **Avaliação de Balanceamento das Classes**
   - 5.1 Distribuição Absoluta e Relativa por Split
   - 5.2 Diagnóstico de Desbalanceamento e Estratégias Recomendadas
6. **Curvas de Densidade por Classe**
   - 6.1 Top Features por Effect Size
   - 6.2 Análise de Sobreposição (Coeficiente de Bhattacharyya)
7. **Tabela Final de Atributos**
8. **Dataset Final e Persistência**
9. **Síntese Final do Pipeline Completo**
 
> **Nota sobre data leakage:** assim como nos entregáveis anteriores, toda estatística dependente de dados (parâmetros de distribuição, testes de hipótese, VIF) é calculada exclusivamente sobre o conjunto de treino (folds 1–8) e aplicada ao dataset completo. A separação treino/validação/teste é mantida intacta até a fase de RP.

---
 
## 1. Importações, Configurações e Dependências
 
Bibliotecas específicas ou de uso mais intensivo neste notebook:
 
- **`statsmodels.stats.outliers_influence.variance_inflation_factor`:** cálculo direto do VIF para cada feature de um conjunto. Quantifica o grau de colinearidade de cada variável em relação às demais.
- **`statsmodels.multivariate.manova.MANOVA`:** teste MANOVA para avaliação global da separabilidade multivariada. Complementa os testes univariados por considerar as features conjuntamente.
- **`sklearn.discriminant_analysis.LinearDiscriminantAnalysis`:** utilizado aqui exclusivamente como ferramenta de projeção 2D para visualização da separabilidade — não como classificador final.
- **`scipy.stats`:** Kruskal-Wallis, Mann-Whitney U e cálculo de eta-quadrado como medida de effect size.
- **`statsmodels.stats.multitest.multipletests`:** correção de Bonferroni e FDR (Benjamini-Hochberg) para o volume de testes realizados nas comparações par-a-par.

In [ ]:
import os
import ast
import gc
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from itertools import combinations
 
# Estatística
from scipy.stats import kruskal, mannwhitneyu, gaussian_kde
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
 
# Análise multivariada (opcional)
try:
    from statsmodels.multivariate.manova import MANOVA
    MANOVA_OK = True
except ImportError:
    MANOVA_OK = False
    print("Aviso: MANOVA nao disponivel nesta versao do statsmodels.")
 
# Scikit-learn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
 
from IPython.display import display, Markdown
 
warnings.filterwarnings('ignore')
 
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size']      = 12
 
np.random.seed(42)

---
 
## 2. Carregamento e Inspeção do Dataset Selecionado

In [ ]:
FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL     = 9
FOLD_TEST    = 10
 
DIR_IN_D9 = Path('../../entregavel-9/outputs/')
FIGS_DIR  = Path('../figuras/')
OUT_DIR   = Path('../outputs/')
 
for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)
 
META_COLS = ['ecg_id', 'patient_id', 'strat_fold', 'quality_class',
             'superclasses_clean', 'primary_class', 'n_superclasses', 'split']
 
parquet_path = DIR_IN_D9 / 'features_selected.parquet'
if not parquet_path.exists():
    raise FileNotFoundError(
        f'Arquivo nao encontrado: {parquet_path}\n'
        'Execute o Entregavel 9 antes de prosseguir.'
    )
 
print('Carregando features selecionadas do Entregavel 9...')
df = pd.read_parquet(str(parquet_path))
 
if 'superclasses_clean' in df.columns and isinstance(df['superclasses_clean'].iloc[0], str):
    df['superclasses_clean'] = df['superclasses_clean'].apply(ast.literal_eval)
 
if 'primary_class' not in df.columns:
    df['primary_class'] = df['superclasses_clean'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else 'UNKNOWN'
    )
 
META_COLS    = [c for c in META_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in META_COLS]
mask_treino  = df['strat_fold'].isin(FOLDS_TREINO)
 
print(f'Dataset carregado    : {df.shape}')
print(f'Features selecionadas: {len(feature_cols)}')
print(f'Registros de treino  : {mask_treino.sum()}')
print(f'Registros de val     : {(df["strat_fold"] == FOLD_VAL).sum()}')
print(f'Registros de teste   : {(df["strat_fold"] == FOLD_TEST).sum()}')
print(f'Registros totais     : {len(df)}')

In [ ]:
# Inspeção básica: NaN, constantes e distribuição de classes
 
df_treino = df[mask_treino].copy()
 
print('Distribuição de classes no conjunto de treino:')
dist_classes = (
    df_treino['primary_class']
    .value_counts()
    .rename_axis('Classe')
    .reset_index(name='N')
    .assign(Percentual=lambda x: (x['N'] / x['N'].sum() * 100).round(2))
)
display(dist_classes)
 
print()
print(f'Features com NaN (treino)  : {df_treino[feature_cols].isnull().sum().sum()}')
print(f'Features constantes        : {(df_treino[feature_cols].std() == 0).sum()}')
print(f'Tipos de dados             : {df_treino[feature_cols].dtypes.unique()}')

In [ ]:
# Padronização para os testes estatísticos (fit exclusivamente no treino)
# O dataset do E9 já vem com RobustScaler aplicado, mas adicionamos StandardScaler
# aqui para garantir variância unitária — necessário para VIF e LDA
 
le      = LabelEncoder()
scaler  = StandardScaler()
 
X_treino_raw = df_treino[feature_cols].values
y_treino     = df_treino['primary_class'].values
y_enc        = le.fit_transform(y_treino)
 
X_treino = scaler.fit_transform(X_treino_raw)
X_todos  = scaler.transform(df[feature_cols].values)
 
CLASSES_ORDER = list(le.classes_)
N_CLASSES     = len(CLASSES_ORDER)
 
print(f'X_treino : {X_treino.shape}')
print(f'Classes  : {CLASSES_ORDER}')

---
 
## Seção 3 — Verificação de Multicolinearidade (VIF)
 
### Fundamentação
 
A **multicolinearidade** ocorre quando duas ou mais features são altamente correlacionadas entre si — isto é, uma delas pode ser aproximadamente expressa como combinação linear das demais. Isso não afeta diretamente a capacidade preditiva de modelos como Random Forest ou SVM-RBF, mas tem consequências concretas em outros contextos:
 
- **Modelos lineares (Regressão Logística, LDA, LASSO):** coeficientes estimados tornam-se instáveis, com alta variância entre amostras de treino. O modelo "não sabe" como distribuir o peso entre features colineares, o que pode inverter o sinal de coeficientes e comprometer a interpretabilidade.
- **Feature importance por impureza (Random Forest):** features colineares "dividem" entre si a importância que seria atribuída a uma única feature informativa, subestimando o papel de cada uma.
- **VIF (Variance Inflation Factor):** quantifica o quanto a variância do coeficiente de uma feature aumenta devido à colinearidade com as demais. Formalmente, para a feature $j$:
 
$$\text{VIF}_j = \frac{1}{1 - R_j^2}$$
 
onde $R_j^2$ é o coeficiente de determinação da regressão de $x_j$ contra todas as outras features. $\text{VIF} = 1$ indica ausência total de colinearidade; $\text{VIF} > 5$ é considerado preocupante; $\text{VIF} > 10$ indica colinearidade severa.
 
É importante notar que a Seleção de Atributos do Entregável 9 já removeu parte da redundância. No entanto, os métodos de seleção (ANOVA, MI, LASSO, RF) não garantem necessariamente a remoção de features colineares — eles selecionam pelo poder discriminativo, não pela independência estrutural. A verificação de VIF aqui é portanto complementar e focada em identificar eventuais resíduos de colinearidade que escaparam da seleção.
 
### 3.1 Cálculo do Variance Inflation Factor

In [ ]:
# Cálculo do VIF para o conjunto de features selecionadas (sobre o treino padronizado)
 
def calcular_vif(X, feature_names):
    """
    Calcula o VIF para cada feature do array X.
    Retorna DataFrame ordenado por VIF decrescente.
    """
    vif_vals = []
    for i in range(X.shape[1]):
        vif_vals.append(variance_inflation_factor(X, i))
 
    df_vif = pd.DataFrame({
        'feature': feature_names,
        'VIF'    : vif_vals
    }).sort_values('VIF', ascending=False).reset_index(drop=True)
 
    return df_vif
 
print('Calculando VIF para todas as features selecionadas...')
df_vif = calcular_vif(X_treino, feature_cols)
 
print()
print('Distribuição dos valores de VIF:')
print(df_vif['VIF'].describe().round(2))
print()
 
vif_severo   = (df_vif['VIF'] > 10).sum()
vif_moderado = ((df_vif['VIF'] > 5) & (df_vif['VIF'] <= 10)).sum()
vif_ok       = (df_vif['VIF'] <= 5).sum()
 
print(f'Features com VIF > 10  (severo)   : {vif_severo}')
print(f'Features com VIF 5-10  (moderado) : {vif_moderado}')
print(f'Features com VIF <= 5  (aceitavel): {vif_ok}')
print()
print('Top 20 features por VIF:')
display(df_vif.head(20).round(3))

In [ ]:
# Visualização da distribuição de VIF
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
# Histograma dos VIFs
axes[0].hist(df_vif['VIF'].clip(upper=50), bins=30,
             color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(5,  color='orange',  ls='--', lw=1.5, label='Limiar moderado (VIF=5)')
axes[0].axvline(10, color='crimson', ls='--', lw=1.5, label='Limiar severo (VIF=10)')
axes[0].set_xlabel('VIF (clipado em 50 para visualizacao)')
axes[0].set_ylabel('Contagem de features')
axes[0].set_title('Distribuicao do VIF — Features Selecionadas (E9)')
axes[0].legend(fontsize=9)
 
# Barplot horizontal — Top 30 por VIF
top30_vif = df_vif.head(30)
cores_vif = ['#e74c3c' if v > 10 else ('#f39c12' if v > 5 else '#2ecc71')
             for v in top30_vif['VIF']]
 
axes[1].barh(top30_vif['feature'][::-1], top30_vif['VIF'][::-1],
             color=cores_vif[::-1], edgecolor='white', linewidth=0.5)
axes[1].axvline(5,  color='orange',  ls='--', lw=1.2)
axes[1].axvline(10, color='crimson', ls='--', lw=1.2)
axes[1].set_xlabel('VIF')
axes[1].set_title('Top 30 Features por VIF\n(vermelho > 10, laranja > 5, verde <= 5)')
 
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_vif_distribuicao.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Heatmap de Correlação Residual

In [ ]:
# Heatmap de correlação entre as features com VIF mais alto
 
N_HEATMAP   = min(40, len(feature_cols))
feats_hmap  = df_vif.head(N_HEATMAP)['feature'].tolist()
corr_resid  = df_treino[feats_hmap].corr(method='pearson')
 
fig, ax = plt.subplots(figsize=(14, 12))
mask_diag = np.eye(len(feats_hmap), dtype=bool)
 
sns.heatmap(
    corr_resid,
    mask=mask_diag,
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.3,
    annot=False,
    ax=ax
)
ax.set_title(
    f'Correlacao de Pearson — Top {N_HEATMAP} features por VIF\n'
    '(Dataset pos-selecao E9, conjunto de treino)',
    fontsize=11
)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_heatmap_correlacao_residual.png', dpi=150, bbox_inches='tight')
plt.show()
 
# Identifica pares com |r| > 0.85 que restaram pós-seleção
i_idx, j_idx = np.triu_indices_from(corr_resid.values, k=1)
pares_altos = [
    (feats_hmap[i], feats_hmap[j], round(corr_resid.values[i, j], 3))
    for i, j in zip(i_idx, j_idx)
    if abs(corr_resid.values[i, j]) > 0.85
]
print(f'Pares com |r| > 0.85 no top {N_HEATMAP} por VIF: {len(pares_altos)}')
if pares_altos:
    df_pares = pd.DataFrame(pares_altos, columns=['feature_A', 'feature_B', 'r_pearson'])
    display(df_pares.sort_values('r_pearson', ascending=False).head(20).round(3))

### 3.3 Decisão e Remoção Iterativa de Features Colineares
 
A abordagem adotada é a **remoção iterativa por limiar de VIF**: em cada iteração, a feature com o VIF mais alto é removida, o VIF é recalculado para o conjunto restante e o processo se repete até que todas as features restantes estejam abaixo do limiar definido ($\text{VIF} \leq 10$).
 
Essa abordagem é preferível à remoção em lote porque a colinearidade é uma propriedade emergente do conjunto — remover uma feature pode reduzir o VIF de outras que dependiam dela estruturalmente.

In [ ]:
VIF_LIMIAR = 10
 
def remocao_iterativa_vif(X, feature_names, limiar=10, verbose=True):
    """
    Remove iterativamente a feature com maior VIF enquanto VIF_max > limiar.
    Retorna lista de features sobreviventes e historico de remocoes.
    """
    features_ativas = list(feature_names)
    X_ativo         = X.copy()
    historico       = []
 
    iteracao = 0
    while True:
        vif_vals = [variance_inflation_factor(X_ativo, i)
                    for i in range(X_ativo.shape[1])]
        max_vif  = max(vif_vals)
        idx_max  = int(np.argmax(vif_vals))
 
        if max_vif <= limiar:
            break
 
        feat_removida = features_ativas[idx_max]
        historico.append({'iteracao'        : iteracao + 1,
                          'feature_removida': feat_removida,
                          'VIF_no_momento'  : round(max_vif, 3)})
        if verbose:
            print(f'  Iter {iteracao+1:3d} | Removendo: {feat_removida:50s} | VIF = {max_vif:.2f}')
 
        features_ativas.pop(idx_max)
        X_ativo = np.delete(X_ativo, idx_max, axis=1)
        iteracao += 1
 
    return features_ativas, X_ativo, pd.DataFrame(historico)
 
print(f'Iniciando remocao iterativa de features com VIF > {VIF_LIMIAR}...')
print(f'Features antes da remocao: {len(feature_cols)}')
print()
 
feats_pos_vif, X_pos_vif, df_hist_vif = remocao_iterativa_vif(
    X_treino, feature_cols, limiar=VIF_LIMIAR
)
 
print()
print(f'Features apos remocao VIF: {len(feats_pos_vif)}')
print(f'Features removidas       : {len(feature_cols) - len(feats_pos_vif)}')
if len(df_hist_vif) > 0:
    print('\nHistorico de remocoes:')
    display(df_hist_vif)

In [ ]:
# Verificacao final de VIF pos-remocao
 
df_vif_final = calcular_vif(X_pos_vif, feats_pos_vif)
 
print('VIF pos-remocao:')
print(df_vif_final['VIF'].describe().round(3))
print()
print(f'Nenhuma feature com VIF > {VIF_LIMIAR}: {(df_vif_final["VIF"] > VIF_LIMIAR).sum() == 0}')
print()
print('Top 10 por VIF (pos-remocao):')
display(df_vif_final.head(10).round(3))
 
# Atualiza o conjunto de features para as secoes seguintes
feature_cols_final = feats_pos_vif
X_treino_final     = X_pos_vif
X_todos_final      = scaler.transform(df[feature_cols_final].values)
 
print(f'\nConjunto final de features: {len(feature_cols_final)}')

**Comentários sobre a Seção 3 — Verificação de Multicolinearidade (VIF):**
 
> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quantas features foram removidas pelo critério VIF > 10? A remoção foi necessária ou o E9 já havia controlado bem a redundância?
> - Quais domínios concentraram as features removidas? Features derivadas de razões espectrais ou deltas inter-derivações tendem a ser mais colineares por construção.
> - O heatmap de correlação residual revela algum padrão por grupo de derivações? Features calculadas para derivações adjacentes (ex: V4, V5, V6) naturalmente apresentam correlações mais altas.
> - O VIF máximo pós-remoção ficou próximo do limiar ou bem abaixo? Isso indica se o processo convergiu com folga ou na margem.

---
 
## Seção 4 — Separabilidade Estatística das Classes
 
### Fundamentação
 
Verificar a separabilidade estatística das classes antes de treinar qualquer classificador é uma etapa frequentemente pulada — e frequentemente lamentada depois. Um dataset onde as classes não diferem estatisticamente em nenhuma feature é, por definição, um problema insolúvel para qualquer modelo supervisionado. Identificar isso agora evita horas de tuning de hiperparâmetros sobre algo fundamentalmente limitado pelos dados.
 
A análise de separabilidade é conduzida em quatro níveis complementares:
 
1. **Kruskal-Wallis por feature:** teste não-paramétrico que avalia se as distribuições de pelo menos duas classes diferem para cada feature individualmente. É o equivalente não-paramétrico do ANOVA F-test, mas sem assumir normalidade ou homocedasticidade — condições que sabemos ser violadas em parte do dataset.
2. **Comparações par-a-par (Mann-Whitney U):** após identificar as features significativas pelo Kruskal-Wallis, investigamos *quais pares de classes* diferem para cada feature. Isso mapeia a estrutura de separabilidade: onde a distinção NORM vs. MI é mais forte? Onde CD e HYP se confundem mais?
3. **Effect size (Eta-quadrado):** p-values são sensíveis ao tamanho amostral — com dezenas de milhares de registros de treino, diferenças minúsculas podem atingir significância estatística sem qualquer relevância prática. O eta-quadrado traduz a diferença estatística em magnitude real.
4. **Projeção LDA 2D:** visualização da separabilidade em um espaço comprimido que maximiza a separação entre classes, permitindo identificar visualmente quais superclasses se agrupam e quais se sobrepõem.
 
### 4.1 Kruskal-Wallis por Feature

In [ ]:
# Kruskal-Wallis para todas as features do conjunto final
# H0: a distribuicao da feature e identica para todas as classes
# Ha: pelo menos uma classe tem distribuicao diferente
 
print(f'Rodando Kruskal-Wallis para {len(feature_cols_final)} features...')
 
grupos_por_classe = {
    cls: df_treino[df_treino['primary_class'] == cls][feature_cols_final].values
    for cls in CLASSES_ORDER
}
 
kw_results = []
for i, feat in enumerate(feature_cols_final):
    grupos = [grupos_por_classe[cls][:, i] for cls in CLASSES_ORDER]
    try:
        H, p = kruskal(*grupos)
    except Exception:
        H, p = np.nan, 1.0
    kw_results.append({'feature': feat, 'H_statistic': H, 'p_value': p})
 
df_kw = pd.DataFrame(kw_results).sort_values('H_statistic', ascending=False).reset_index(drop=True)
 
# Correcao de Bonferroni e FDR
_, p_bonf, _, _ = multipletests(df_kw['p_value'].fillna(1.0), method='bonferroni')
_, p_fdr,  _, _ = multipletests(df_kw['p_value'].fillna(1.0), method='fdr_bh')
 
df_kw['p_bonferroni']      = p_bonf
df_kw['p_fdr_bh']          = p_fdr
df_kw['significativa_fdr'] = df_kw['p_fdr_bh'] < 0.05
df_kw['rank_kw']           = df_kw.index + 1
 
sig_fdr  = df_kw['significativa_fdr'].sum()
sig_bonf = (df_kw['p_bonferroni'] < 0.05).sum()
 
print()
print(f'Features significativas (p_FDR < 0.05)       : {sig_fdr} de {len(feature_cols_final)}')
print(f'Features significativas (p_Bonferroni < 0.05): {sig_bonf} de {len(feature_cols_final)}')
print()
print('Top 20 features por H-statistic:')
display(df_kw.head(20).round(4))

In [ ]:
# Visualizacao: distribuicao do H-statistic e top 30
 
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
 
# Histograma
axes[0].hist(df_kw['H_statistic'].dropna(), bins=40,
             color='#2980b9', edgecolor='white', alpha=0.85)
axes[0].set_xlabel('H-statistic (Kruskal-Wallis)')
axes[0].set_ylabel('Contagem de features')
axes[0].set_title('Distribuicao do H-statistic\n(todas as features, conjunto de treino)')
h_med = df_kw['H_statistic'].median()
axes[0].axvline(h_med, color='crimson', ls='--', lw=1.5,
                label=f'Mediana: {h_med:.1f}')
axes[0].legend(fontsize=9)
 
# Barplot top 30
top30_kw = df_kw.head(30)
cores_kw = ['#27ae60' if s else '#bdc3c7' for s in top30_kw['significativa_fdr']]
axes[1].barh(top30_kw['feature'][::-1], top30_kw['H_statistic'][::-1],
             color=cores_kw[::-1], edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('H-statistic')
axes[1].set_title('Top 30 Features — Kruskal-Wallis\n(verde = significativa p_FDR < 0.05)')
 
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_kruskal_wallis.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Comparações Par-a-Par (Mann-Whitney U)

In [ ]:
# Mann-Whitney U — comparacoes par-a-par nas top K features por H-statistic
 
TOP_K_PAR = min(30, len(feature_cols_final))
feats_par = df_kw.head(TOP_K_PAR)['feature'].tolist()
pares     = list(combinations(CLASSES_ORDER, 2))
 
print(f'Comparacoes par-a-par: top {TOP_K_PAR} features x {len(pares)} pares de classes...')
 
rows_mw = []
for feat in feats_par:
    vals_por_classe = {
        cls: df_treino[df_treino['primary_class'] == cls][feat].dropna().values
        for cls in CLASSES_ORDER
    }
    for cls_a, cls_b in pares:
        a, b = vals_por_classe[cls_a], vals_por_classe[cls_b]
        try:
            U, p = mannwhitneyu(a, b, alternative='two-sided')
            n_a, n_b = len(a), len(b)
            r_rb = abs(1 - (2 * U) / (n_a * n_b))
        except Exception:
            p, r_rb = 1.0, 0.0
        rows_mw.append({'feature'  : feat,
                        'classe_A' : cls_a,
                        'classe_B' : cls_b,
                        'p_value'  : p,
                        'r_rb'     : r_rb})
 
df_mw = pd.DataFrame(rows_mw)
_, p_fdr_mw, _, _ = multipletests(df_mw['p_value'], method='fdr_bh')
df_mw['p_fdr'] = p_fdr_mw
df_mw['sig']   = df_mw['p_fdr'] < 0.05
 
# Pivot: n features que separam significativamente cada par
pivot_sig = (
    df_mw[df_mw['sig']]
    .groupby(['classe_A', 'classe_B'])['feature']
    .count()
    .reset_index(name='n_features_sig')
)
pivot_sig['par'] = pivot_sig['classe_A'] + ' vs ' + pivot_sig['classe_B']
 
print()
print('Separabilidade por par de classes (n features com p_FDR < 0.05):')
display(pivot_sig[['par', 'n_features_sig']].sort_values('n_features_sig', ascending=False))

In [ ]:
# Heatmap de separabilidade par-a-par
 
pivot_matrix = pd.DataFrame(
    np.zeros((N_CLASSES, N_CLASSES), dtype=int),
    index=CLASSES_ORDER, columns=CLASSES_ORDER
)
for _, row in pivot_sig.iterrows():
    pivot_matrix.loc[row['classe_A'], row['classe_B']] = row['n_features_sig']
    pivot_matrix.loc[row['classe_B'], row['classe_A']] = row['n_features_sig']
 
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    pivot_matrix,
    annot=True, fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
    ax=ax
)
ax.set_title(
    f'Separabilidade Par-a-Par (Mann-Whitney U, p_FDR < 0.05)\n'
    f'Numero de features que separam significativamente cada par de classes',
    fontsize=10
)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_separabilidade_par_a_par.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Effect Size — Eta-quadrado (η²)
 
O **Eta-quadrado (η²)** estima a proporção da variância total da feature que é explicada pela pertença de cada instância à sua respectiva classe. É calculado a partir do H-statistic do Kruskal-Wallis:
 
$$\eta^2 = \frac{H - k + 1}{n - k}$$
 
onde $H$ é o H-statistic, $k$ é o número de classes e $n$ é o total de instâncias no treino.
 
Referências (Cohen, 1988): η² < 0,01 → negligenciável; 0,01–0,06 → pequeno; 0,06–0,14 → médio; > 0,14 → grande.

In [ ]:
# Eta-quadrado a partir do H-statistic
 
n_treino = mask_treino.sum()
k        = N_CLASSES
 
df_kw['eta_sq'] = (df_kw['H_statistic'] - k + 1) / (n_treino - k)
df_kw['eta_sq'] = df_kw['eta_sq'].clip(lower=0)
 
def cat_eta(e):
    if e < 0.01 : return 'negligivel'
    if e < 0.06 : return 'pequeno'
    if e < 0.14 : return 'medio'
    return 'grande'
 
df_kw['effect_size_cat'] = df_kw['eta_sq'].apply(cat_eta)
 
print('Distribuicao de effect size (eta-quadrado):')
display(df_kw['effect_size_cat'].value_counts().to_frame('n_features'))
print()
print('Top 20 por eta-quadrado:')
display(
    df_kw[['feature', 'H_statistic', 'p_fdr_bh', 'eta_sq', 'effect_size_cat']]
    .sort_values('eta_sq', ascending=False)
    .head(20)
    .round(4)
)

In [ ]:
# Scatter: H-statistic vs. eta-quadrado, colorido por category de effect size
 
fig, ax = plt.subplots(figsize=(10, 6))
 
paleta_es = {'negligivel': '#bdc3c7', 'pequeno': '#85c1e9',
             'medio': '#f0b27a', 'grande': '#e74c3c'}
 
for cat, grp in df_kw.groupby('effect_size_cat'):
    ax.scatter(grp['H_statistic'], grp['eta_sq'],
               color=paleta_es.get(cat, 'gray'),
               alpha=0.6, s=20, label=cat.capitalize())
 
ax.set_xlabel('H-statistic (Kruskal-Wallis)')
ax.set_ylabel('Eta-quadrado (eta²)')
ax.set_title('Effect Size vs. H-statistic por Feature\n(conjunto de treino)')
ax.legend(title='Effect size', fontsize=9)
ax.axhline(0.14, color='crimson', ls=':', lw=1.2)
ax.axhline(0.06, color='orange',  ls=':', lw=1.2)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_effect_size_eta_sq.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Análise Discriminante Visual — Projeção LDA 2D

In [ ]:
# Projecao LDA 2D — visualizacao da separabilidade no espaco discriminante
 
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_treino_final, y_enc)
 
var_exp = lda.explained_variance_ratio_
print(f'Variancia explicada — LD1: {var_exp[0]*100:.1f}%  |  LD2: {var_exp[1]*100:.1f}%')
print(f'Total (LD1+LD2): {sum(var_exp)*100:.1f}%')
 
cores_lda = sns.color_palette('Set2', N_CLASSES)
fig, ax   = plt.subplots(figsize=(10, 8))
 
for idx, cls in enumerate(CLASSES_ORDER):
    mask_cls = y_enc == idx
    ax.scatter(X_lda[mask_cls, 0], X_lda[mask_cls, 1],
               color=cores_lda[idx], label=cls,
               alpha=0.35, s=8, linewidths=0)
 
# Centroides
for idx, cls in enumerate(CLASSES_ORDER):
    mask_cls = y_enc == idx
    cx, cy = X_lda[mask_cls, 0].mean(), X_lda[mask_cls, 1].mean()
    ax.scatter(cx, cy, color=cores_lda[idx], s=200, marker='*',
               edgecolors='black', linewidths=0.7, zorder=5)
    ax.annotate(cls, (cx, cy), textcoords='offset points',
                xytext=(6, 4), fontsize=10, fontweight='bold',
                color=cores_lda[idx])
 
ax.set_xlabel(f'LD1 ({var_exp[0]*100:.1f}% da variancia explicada)')
ax.set_ylabel(f'LD2 ({var_exp[1]*100:.1f}% da variancia explicada)')
ax.set_title('Projecao LDA 2D — Separabilidade das Superclasses\n'
             '(conjunto de treino | estrelas = centroides)')
ax.legend(title='Superclasse', fontsize=9, markerscale=3)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_lda_2d_separabilidade.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a Seção 4 — Separabilidade Estatística:**
 
> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quantas features passaram no Kruskal-Wallis com p_FDR < 0.05? Isso representa qual proporção do total selecionado?
> - Quais pares de classes são mais difíceis de separar (menos features com diferença significativa no heatmap)? MI vs. STTC costuma ser o par mais desafiador no PTB-XL — confirma-se aqui?
> - Qual a distribuição de effect size? Predominância de efeitos médios/grandes indica que o pipeline capturou diferenças fisiológicas reais.
> - No LDA 2D: NORM e HYP tendem a ser as classes mais separadas. MI, STTC e CD frequentemente se sobrepõem. O gráfico confirma esse padrão esperado?
> - Os centroides estão bem distribuídos no espaço LD1×LD2 ou há classes muito próximas? Isso antecipa quais pares o classificador vai errar mais.

---
 
## Seção 5 — Avaliação de Balanceamento das Classes
 
### Fundamentação
 
O desbalanceamento entre classes é um dos problemas mais práticos e frequentemente subestimados em projetos de aprendizado supervisionado. No PTB-XL, a distribuição das superclasses reflete o mundo real: condições mais comuns (NORM, STTC) são naturalmente mais representadas do que condições menos frequentes (HYP). Isso cria um cenário em que um classificador ingênuo poderia atingir alta acurácia simplesmente predizendo sempre a classe majoritária.
 
Documentamos formalmente a distribuição absoluta e relativa por split, o grau de desbalanceamento via **Imbalance Ratio (IR)** e o **índice de Shannon normalizado** da distribuição de classes, além das estratégias recomendadas para o componente de RP.

In [ ]:
# Distribuicao por split e classe
 
splits_map = {
    'treino'    : df['strat_fold'].isin(FOLDS_TREINO),
    'validacao' : df['strat_fold'] == FOLD_VAL,
    'teste'     : df['strat_fold'] == FOLD_TEST,
}
 
rows_bal = []
for split_name, mask_split in splits_map.items():
    for cls in CLASSES_ORDER:
        n = (df[mask_split]['primary_class'] == cls).sum()
        rows_bal.append({'split': split_name, 'classe': cls, 'n': n})
 
df_bal = pd.DataFrame(rows_bal)
df_bal['pct'] = df_bal.groupby('split')['n'].transform(
    lambda x: (x / x.sum() * 100).round(2)
)
 
print('Distribuicao de classes por split:')
display(
    df_bal.pivot_table(index='classe', columns='split', values=['n', 'pct'])
    .round(2)
)

In [ ]:
from scipy.stats import entropy as shannon_entropy
 
print('Metricas de desbalanceamento por split:\n')
for split_name, mask_split in splits_map.items():
    contagem = df[mask_split]['primary_class'].value_counts()
    n_max    = contagem.max()
    n_min    = contagem.min()
    ir       = n_max / n_min
    probs    = contagem.values / contagem.sum()
    h_norm   = shannon_entropy(probs) / np.log(N_CLASSES)  # entropia normalizada [0, 1]
 
    status = ('BALANCEADO' if ir < 1.5 else
              ('MODERADO'  if ir < 3.0 else 'DESBALANCEADO'))
 
    print(f'  {split_name.upper():12s} | IR = {ir:.2f} | H_norm = {h_norm:.3f} | {status}')
    print(f'               | Classe maior: {contagem.idxmax()} (n={n_max})')
    print(f'               | Classe menor: {contagem.idxmin()} (n={n_min})')
    print()

In [ ]:
# Graficos de barras agrupadas por split
 
fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
 
for ax, (split_name, mask_split) in zip(axes, splits_map.items()):
    contagem = df[mask_split]['primary_class'].value_counts().reindex(CLASSES_ORDER)
    cores    = sns.color_palette('Set2', N_CLASSES)
 
    bars = ax.bar(CLASSES_ORDER, contagem.values,
                  color=cores, edgecolor='white', linewidth=0.7)
    ax.set_title(f'Split: {split_name.capitalize()}\n(N = {contagem.sum()})', fontsize=11)
    ax.set_xlabel('Superclasse')
    ax.set_ylabel('Numero de registros')
 
    for bar, val in zip(bars, contagem.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontsize=9)
 
plt.suptitle('Distribuicao de Classes por Split — Dataset Final', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_balanceamento_por_split.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Verificacao de estratificacao por fold (proporcao de NORM por fold)
 
print('Proporcao de NORM por fold (indicador de estratificacao correta):')
for fold in FOLDS_TREINO + [FOLD_VAL, FOLD_TEST]:
    mask_fold = df['strat_fold'] == fold
    pct_norm  = (df[mask_fold]['primary_class'] == 'NORM').mean() * 100
    tag = 'treino' if fold in FOLDS_TREINO else ('val' if fold == FOLD_VAL else 'teste')
    print(f'  Fold {fold:2d} ({tag:6s}) | NORM: {pct_norm:.1f}%  | N = {mask_fold.sum()}')

### 5.2 Diagnóstico e Estratégias Recomendadas para o RP
 
Com base nos valores de IR calculados acima, as recomendações para o componente de RP são:
 
| Condição | Estratégia Recomendada |
|---|---|
| IR < 1,5 (balanceado) | Nenhuma ação especial. Acurácia pode ser usada como métrica principal. |
| 1,5 ≤ IR < 3,0 (moderado) | Usar `class_weight='balanced'` nos classificadores. Preferir F1-macro como métrica. |
| IR ≥ 3,0 (desbalanceado) | Considerar SMOTE no treino. Usar F1-macro / AUC-ROC. Avaliar recall por classe separadamente. |
 
> **Nota importante:** qualquer técnica de reamostramento (SMOTE, oversampling, undersampling) deve ser aplicada **exclusivamente dentro do conjunto de treino**, após a separação treino/validação/teste, e nunca antes. Aplicar antes da separação constitui data leakage e inflaciona artificialmente as métricas de validação.

---
 
## Seção 6 — Curvas de Densidade por Classe
 
### Fundamentação
 
As curvas de densidade (KDE — Kernel Density Estimate) por classe são a representação visual mais direta da separabilidade univariada. Para cada feature, plotamos as distribuições de probabilidade estimadas por kernel gaussiano para cada superclasse. A sobreposição visual entre curvas traduz de forma imediata o quanto aquela feature consegue discriminar as classes:
 
- **Curvas bem separadas:** a feature é discriminativa e provavelmente aparecerá com importância alta nos classificadores.
- **Curvas com sobreposição parcial:** discriminação dependente de ponto de corte — útil em combinação com outras features.
- **Curvas completamente sobrepostas:** feature com pouco poder discriminativo isolada — útil apenas via interações.
 
Plotamos as features com maior η², organizadas por domínio de origem, garantindo cobertura de todos os grupos de atributos.
 
### 6.1 KDE — Top Features por Domínio

In [ ]:
# Mapeamento de dominio
 
def get_domain(col):
    if col.startswith('time_')         : return 'Tempo'
    if col.startswith('freq_')         : return 'Frequencia'
    if col.startswith('hrv_')          : return 'HRV'
    if col.startswith('morph_')        : return 'Morfologia'
    if col.startswith('wavelet_')      : return 'Wavelet'
    if col.startswith('nonlin_')       : return 'Nao-Linear'
    if col.startswith('ratio_')        : return 'Razao Espectral'
    if col.startswith('norm_baseline_'): return 'Baseline NORM'
    if col.startswith('delta_')        : return 'Delta Inter-Deriv.'
    return 'Outro'
 
# Filtra o df_kw para as features do conjunto final e adiciona dominio
df_kw_final = df_kw[df_kw['feature'].isin(feature_cols_final)].copy()
df_kw_final['dominio'] = df_kw_final['feature'].apply(get_domain)
 
# Top 2 por dominio (por eta_sq)
feats_density = (
    df_kw_final
    .sort_values('eta_sq', ascending=False)
    .groupby('dominio', group_keys=False)
    .head(2)
    ['feature']
    .tolist()
)
 
feats_density = feats_density[:16]
n_feats_dens  = len(feats_density)
n_cols        = 4
n_rows        = int(np.ceil(n_feats_dens / n_cols))
 
cores_kde = dict(zip(CLASSES_ORDER, sns.color_palette('Set2', N_CLASSES)))
print(f'Plotando curvas de densidade para {n_feats_dens} features ({n_rows} x {n_cols} grid)...')

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()
 
for ax, feat in zip(axes, feats_density):
    eta_vals = df_kw_final[df_kw_final['feature'] == feat]['eta_sq'].values
    eta_str  = f'eta²={eta_vals[0]:.3f}' if len(eta_vals) > 0 else ''
    dom_feat = get_domain(feat)
 
    for cls in CLASSES_ORDER:
        vals = df_treino[df_treino['primary_class'] == cls][feat].dropna().values
        if len(vals) < 10:
            continue
        kde    = gaussian_kde(vals, bw_method='scott')
        x_grid = np.linspace(vals.min(), vals.max(), 300)
        ax.fill_between(x_grid, kde(x_grid), alpha=0.15, color=cores_kde[cls])
        ax.plot(x_grid, kde(x_grid), lw=1.4, color=cores_kde[cls], label=cls)
 
    ax.set_title(f'{feat}\n{dom_feat} | {eta_str}', fontsize=7.5)
    ax.set_xlabel('Valor (padronizado)')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=6, loc='upper right')
    ax.grid(True, alpha=0.3)
 
for ax in axes[n_feats_dens:]:
    ax.set_visible(False)
 
plt.suptitle('Curvas de Densidade por Classe — Top Features por Dominio\n'
             '(conjunto de treino, KDE gaussiana)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(FIGS_DIR / 'val_kde_por_classe_top_features.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Análise de Sobreposição — Coeficiente de Bhattacharyya

In [ ]:
# Coeficiente de Bhattacharyya: medida de sobreposicao entre duas distribuicoes
# BC = 1.0: distribuicoes identicas  |  BC = 0.0: sem sobreposicao
 
def bhattacharyya_coef(a, b, n_bins=50):
    lo   = min(a.min(), b.min())
    hi   = max(a.max(), b.max())
    bins = np.linspace(lo, hi, n_bins + 1)
    ha, _ = np.histogram(a, bins=bins, density=True)
    hb, _ = np.histogram(b, bins=bins, density=True)
    ha    = ha / (ha.sum() + 1e-12)
    hb    = hb / (hb.sum() + 1e-12)
    return float(np.sum(np.sqrt(ha * hb)))
 
print('Coeficiente de Bhattacharyya — sobreposicao entre NORM e cada patologia')
print('(1.0 = identico; 0.0 = sem sobreposicao)\n')
 
rows_bc = []
for feat in feats_density:
    vals_norm = df_treino[df_treino['primary_class'] == 'NORM'][feat].dropna().values
    for cls in [c for c in CLASSES_ORDER if c != 'NORM']:
        vals_cls = df_treino[df_treino['primary_class'] == cls][feat].dropna().values
        bc = bhattacharyya_coef(vals_norm, vals_cls)
        rows_bc.append({'feature': feat, 'NORM_vs': cls, 'BC': round(bc, 4)})
 
df_bc = pd.DataFrame(rows_bc)
pivot_bc = df_bc.pivot(index='feature', columns='NORM_vs', values='BC').round(3)
display(pivot_bc)
 
# Destaca os pares com menor separacao (BC mais alto)
bc_medio = df_bc.groupby('NORM_vs')['BC'].mean().sort_values(ascending=False)
print()
print('BC medio por classe (NORM vs. X) — maior = mais sobreposicao:')
display(bc_medio.rename('BC_medio').reset_index().round(3))

**Comentários sobre a Seção 6 — Curvas de Densidade por Classe:**
 
> 💬 *Preencha aqui com os resultados obtidos. Sugestões do que comentar:*
> - Quais features apresentam as curvas mais bem separadas? Features de energia (wavelet, espectral) costumam separar bem HYP das demais classes.
> - Há casos de bimodalidade dentro de alguma classe? Isso indicaria heterogeneidade intra-classe — possível em MI (diferentes estágios do infarto).
> - O coeficiente de Bhattacharyya confirma os pares mais difíceis identificados pelo Mann-Whitney? A coerência entre as análises reforça a robustez do diagnóstico.
> - Features de qual domínio apresentam menor sobreposição de forma geral? Isso orienta diretamente quais grupos de atributos serão mais relevantes nos classificadores.

---
 
## Seção 7 — Tabela Final de Atributos
 
Esta seção consolida o catálogo completo das features que compõem o dataset final. A tabela inclui nome, domínio de origem, estatísticas descritivas no treino, H-statistic, η² e VIF. Este catálogo é o documento de referência que acompanha o dataset entregue ao componente de RP.

In [ ]:
# Estatisticas descritivas das features no conjunto de treino
 
desc_treino = df_treino[feature_cols_final].agg(['mean', 'std', 'min', 'max']).T
desc_treino.columns = ['media_treino', 'std_treino', 'min_treino', 'max_treino']
desc_treino.index.name = 'feature'
 
# Merge com resultados dos testes e VIF
df_kw_cat = (
    df_kw_final[['feature', 'H_statistic', 'p_fdr_bh', 'eta_sq',
                 'effect_size_cat', 'rank_kw', 'dominio']]
    .set_index('feature')
)
 
df_catalog_final = desc_treino.join(df_kw_cat, how='left')
df_catalog_final = df_catalog_final.join(
    df_vif_final.set_index('feature')[['VIF']], how='left'
)
df_catalog_final = df_catalog_final.sort_values('eta_sq', ascending=False)
 
print(f'Catalogo final: {len(df_catalog_final)} features')
print()
print('Amostra — Top 20 por eta-quadrado:')
display(df_catalog_final.head(20).round(4))

In [ ]:
# Resumo por dominio
 
resumo_dominio = (
    df_catalog_final
    .reset_index()
    .groupby('dominio')
    .agg(
        n_features         = ('feature', 'count'),
        eta_sq_mediana     = ('eta_sq', 'median'),
        eta_sq_max         = ('eta_sq', 'max'),
        pct_efeito_grande  = ('effect_size_cat',
                              lambda x: (x == 'grande').mean() * 100),
        VIF_medio          = ('VIF', 'mean')
    )
    .round(3)
    .sort_values('eta_sq_mediana', ascending=False)
)
 
print('Resumo por dominio:')
display(resumo_dominio)

---
 
## Seção 8 — Dataset Final e Persistência
 
### 8.1 Salvamento dos Artefatos

In [ ]:
# Monta o dataset final com features pos-VIF + metadados originais
 
df_final = pd.concat([
    df[META_COLS].reset_index(drop=True),
    pd.DataFrame(
        scaler.inverse_transform(X_todos_final),
        columns=feature_cols_final
    )
], axis=1)
 
df_final.index = df.index
 
# Caminhos de saida
path_parquet_final = OUT_DIR / 'dataset_final_validado.parquet'
path_csv_sample    = OUT_DIR / 'dataset_final_validado_sample.csv'
path_catalog       = OUT_DIR / 'feature_catalog_e10.csv'
path_rel_vif       = OUT_DIR / 'relatorio_vif.csv'
path_rel_kw        = OUT_DIR / 'relatorio_kruskal_wallis.csv'
path_rel_bal       = OUT_DIR / 'relatorio_balanceamento.csv'
 
df_final.to_parquet(str(path_parquet_final), index=True)
df_final.head(500).to_csv(str(path_csv_sample), index=True)
df_catalog_final.to_csv(str(path_catalog))
df_vif_final.to_csv(str(path_rel_vif), index=False)
df_kw.to_csv(str(path_rel_kw), index=False)
df_bal.to_csv(str(path_rel_bal), index=False)
 
print('Artefatos salvos em:', OUT_DIR.resolve())
print()
print(f'  dataset_final_validado.parquet     — dataset completo ({df_final.shape})')
print(f'  dataset_final_validado_sample.csv  — primeiras 500 linhas')
print(f'  feature_catalog_e10.csv            — catalogo completo de features')
print(f'  relatorio_vif.csv                  — VIF por feature')
print(f'  relatorio_kruskal_wallis.csv        — testes KW e effect size')
print(f'  relatorio_balanceamento.csv         — distribuicao de classes por split')

In [ ]:
# Verificacao final de integridade
 
print('Verificacao de integridade do dataset final:\n')
print(f'  Shape                 : {df_final.shape}')
print(f'  Valores nulos         : {df_final[feature_cols_final].isnull().sum().sum()}')
print(f'  Features constantes   : {(df_final[feature_cols_final].std() == 0).sum()}')
print(f'  Registros treino      : {df_final["strat_fold"].isin(FOLDS_TREINO).sum()}')
print(f'  Registros validacao   : {(df_final["strat_fold"] == FOLD_VAL).sum()}')
print(f'  Registros teste       : {(df_final["strat_fold"] == FOLD_TEST).sum()}')
print()
 
assert list(df_final.index) == list(df.index), 'Indices desalinhados!'
print('  Alinhamento de indices: OK')
print()
print('  Dataset pronto para o componente de Reconhecimento de Padroes.')

---
 
## Seção 9 — Síntese Final do Pipeline Completo
 
### 9.1 O que foi feito neste entregável
 
| Verificação | Resultado |
|---|---|
| VIF — multicolinearidade | Features com VIF > 10 removidas iterativamente; conjunto final sem colinearidade severa |
| Heatmap de correlação residual | Estrutura de correlação pós-seleção documentada |
| Kruskal-Wallis por feature | Proporção de features significativas (p_FDR < 0,05) reportada |
| Effect size (η²) | Distribuição entre categorias (negligível / pequeno / médio / grande) |
| Mann-Whitney par-a-par | Heatmap de separabilidade para todos os pares de classes |
| Projeção LDA 2D | Separabilidade visual das superclasses no espaço discriminante |
| Balanceamento por split | IR e entropia de Shannon documentados; estratégias definidas |
| Curvas de densidade KDE | Sobreposição quantificada via coeficiente de Bhattacharyya |
 
### 9.2 Síntese do Pipeline Completo (Entregáveis 1–10)
 
O pipeline de pré-processamento percorreu dez etapas organizadas em três fases:
 
**Fase 1 — Aquisição e Qualidade do Sinal (E1–E2):** definição do protocolo experimental, configuração da taxa de amostragem segundo o teorema de Nyquist e avaliação formal de qualidade (SQI) com descarte de segmentos inadequados.
 
**Fase 2 — Processamento do Sinal (E3–E5):** análise estatística inicial da base bruta, limpeza e filtragem (notch, band-pass, remoção de artefatos, winsorização com validação antes/depois) e segmentação em janelas com verificação de estabilidade intra e inter-janela.
 
**Fase 3 — Representação e Seleção (E6–E10):** extração de atributos emquatro domínios (tempo, frequência, tempo-frequência e não-linear), engenhariade features com combinações e normalizações fisiologicamente motivadas, ebifurcação em duas trilhas paralelas a partir do E7: (a) redução dedimensionalidade via PCA/ICA (E8), que produz o `features_pca.parquet` comorepresentação comprimida para uso direto no RP; (b) seleção formal deatributos por métodos filter/wrapper/embedded com correção para múltiplascomparações (E9), que preserva as features originais e alimenta o E10.A validação estatística final (E10) opera sobre a trilha interpretável (E9),atestando multicolinearidade, separabilidade, balanceamento e distribuiçãopor classe antes da entrega ao componente de Reconhecimento de Padrões.
 
O resultado é um dataset tabulado, sem valores ausentes, sem multicolinearidade severa, com features estatisticamente validadas e com a estrutura de separabilidade e balanceamento rigorosamente documentada — pronto para alimentar qualquer classificador no componente de Reconhecimento de Padrões.

In [ ]:
# Bloco de resumo final consolidado
 
print('=' * 60)
print('   VERIFICACAO FINAL — ENTREGAVEL 10')
print('=' * 60)
print()
print('  DATASET FINAL:')
print(f'    Shape                   : {df_final.shape}')
print(f'    Features finais         : {len(feature_cols_final)}')
print(f'    Valores ausentes        : 0')
print(f'    Features c/ VIF > 10   : 0  (removidas iterativamente)')
print()
print('  SEPARABILIDADE:')
 
n_sig = df_kw[
    df_kw['feature'].isin(feature_cols_final) &
    df_kw['significativa_fdr']
].shape[0]
ef_grandes = (
    df_kw_final[df_kw_final['feature'].isin(feature_cols_final)]
    ['effect_size_cat']
    .eq('grande')
    .sum()
)
print(f'    Features KW sig. (FDR) : {n_sig} / {len(feature_cols_final)}')
print(f'    Effect size grande     : {ef_grandes} features (eta² > 0.14)')
print()
print('  BALANCEAMENTO:')
ct = df[mask_treino]['primary_class'].value_counts()
ir = ct.max() / ct.min()
print(f'    Imbalance Ratio treino : {ir:.2f}')
print(f'    Classe maior (treino)  : {ct.idxmax()} (n={ct.max()})')
print(f'    Classe menor (treino)  : {ct.idxmin()} (n={ct.min()})')
print()
print('  ARTEFATOS GERADOS:')
print('    - dataset_final_validado.parquet')
print('    - dataset_final_validado_sample.csv')
print('    - feature_catalog_e10.csv')
print('    - relatorio_vif.csv')
print('    - relatorio_kruskal_wallis.csv')
print('    - relatorio_balanceamento.csv')
print()
print('  Dataset pronto para: SVM, kNN, RF, Regressao Logistica, CNN, etc.')
print('=' * 60)